# LOB Visualization: BUY + SELL Combined

Комбинированный анализ aggressive scenario для BUY и SELL.
- Два проигрывателя книг (BUY и SELL)
- Midprice comparison: BUY vs SELL*(-1)
- Combined impact: (BUY + SELL*(-1)) / 2

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

## Data Loading

In [ ]:
# === CONFIGURATION ===
# Set paths to BUY and SELL data folders
DATA_PATH_BUY = Path("/app/output/evalsequences/aggressive_scenario/exp_47_20260205_221310")
DATA_PATH_SELL = Path("/app/output/evalsequences/aggressive_scenario/exp_48_20260205_221314")

# Samples to exclude from analysis (list of sample_ids)
EXCLUDE_SAMPLES_BUY = [1996, 266]
EXCLUDE_SAMPLES_SELL = [1996, 266]
# Example:
# EXCLUDE_SAMPLES_BUY = [4, 5, 1894]
# EXCLUDE_SAMPLES_SELL = [100, 200]


import re

def discover_data_params(data_path):
    """Auto-discover ticker and sample info from files in data_cond folder."""
    cond_dir = data_path / "data_cond"
    pattern = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    
    tickers = set()
    sample_info = {}  # {sample_id: date}
    
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        match = pattern.match(f.name)
        if match:
            tickers.add(match.group(1))
            sample_id = int(match.group(3))
            date = match.group(2)
            sample_info[sample_id] = date
    
    if not sample_info:
        raise ValueError(f"No orderbook files found in {cond_dir}")
    if len(tickers) > 1:
        raise ValueError(f"Multiple tickers found: {tickers}")
    
    ticker = tickers.pop()
    dates = set(sample_info.values())
    
    print(f"Discovered: ticker={ticker}, dates={sorted(dates)}, n_samples={len(sample_info)}")
    return ticker, sample_info


def load_scenario_data(data_path, exclude_samples, scenario_name):
    """Load data for one scenario (BUY or SELL)."""
    print(f"\n=== Loading {scenario_name} data ===")
    ticker, sample_info = discover_data_params(data_path)
    
    books = {}
    msgs = {}
    cond_lens = {}
    
    for sid, date in sorted(sample_info.items()):
        if sid in exclude_samples:
            continue
            
        cond_book = np.loadtxt(data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv", delimiter=',')
        cond_msg = np.loadtxt(data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv", delimiter=',')
        
        gen_book = np.loadtxt(data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv", delimiter=',')
        gen_msg = np.loadtxt(data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv", delimiter=',')
        
        cond_lens[sid] = cond_book.shape[0]
        books[sid] = np.vstack([cond_book, gen_book])
        msgs[sid] = np.vstack([cond_msg, gen_msg])
    
    excluded_count = len([s for s in sample_info.keys() if s in exclude_samples])
    print(f"Loaded {len(books)} samples (excluded {excluded_count})")
    return books, msgs, cond_lens


# Load both scenarios
buy_books, buy_msgs, buy_cond_lens = load_scenario_data(DATA_PATH_BUY, EXCLUDE_SAMPLES_BUY, "BUY")
sell_books, sell_msgs, sell_cond_lens = load_scenario_data(DATA_PATH_SELL, EXCLUDE_SAMPLES_SELL, "SELL")

# Find common samples
common_samples = set(buy_books.keys()) & set(sell_books.keys())
print(f"\n=== Common samples: {len(common_samples)} ===")


=== Loading BUY data ===
Discovered: ticker=GOOG, dates=['2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06', '2023-01-09', '2023-01-10', '2023-01-11', '2023-01-12', '2023-01-13'], n_samples=64
Loaded 64 samples (excluded 0)

=== Loading SELL data ===
Discovered: ticker=GOOG, dates=['2023-01-03', '2023-01-04', '2023-01-05', '2023-01-06', '2023-01-09', '2023-01-10', '2023-01-11', '2023-01-12', '2023-01-13'], n_samples=64
Loaded 64 samples (excluded 0)

=== Common samples: 64 ===


## Helper Functions

In [ ]:
# Load aggressive indices from file (indices within GEN portion)
def load_aggressive_indices(data_path):
    """Load aggressive order indices from aggressive_indices.csv.
    
    These are indices within the GEN portion (after junction).
    Returns empty array if file not found.
    """
    aggr_file = data_path / 'aggressive_indices.csv'
    if aggr_file.exists():
        indices = np.loadtxt(aggr_file, dtype=int)
        # Handle single value case
        if indices.ndim == 0:
            indices = np.array([int(indices)])
        print(f"Loaded aggressive indices from {aggr_file}: {indices}")
        return indices
    else:
        print(f"Warning: {aggr_file} not found")
        return np.array([], dtype=int)

# Load aggressive indices for both scenarios
AGGRESSIVE_INDICES_BUY = load_aggressive_indices(DATA_PATH_BUY)
AGGRESSIVE_INDICES_SELL = load_aggressive_indices(DATA_PATH_SELL)

def extract_quantities(book_row):
    """Extract quantities from orderbook row (LOBSTER interleaved format)."""
    ask_qtys = book_row[1::4]
    bid_qtys = book_row[3::4]
    return np.concatenate([bid_qtys, ask_qtys])

def compute_queued_volumes(book_array):
    """Compute bid/ask/total queued volume over time."""
    T = book_array.shape[0]
    ask_vol = np.array([np.sum(np.abs(book_array[t, 1::4])) for t in range(T)])
    bid_vol = np.array([np.sum(np.abs(book_array[t, 3::4])) for t in range(T)])
    return bid_vol, ask_vol, bid_vol + ask_vol

def compute_midprice(book_array):
    """Compute midprice: (best_ask + best_bid) / 2."""
    return (book_array[:, 0] + book_array[:, 2]) / 2

def compute_midprice_return(book_array):
    """Compute midprice return from first value."""
    mid = compute_midprice(book_array)
    return mid - mid[0]

def get_absolute_aggressive_indices(aggressive_indices_gen, junction):
    """Convert GEN-relative indices to absolute indices in full array.
    
    Args:
        aggressive_indices_gen: indices within GEN portion (from aggressive_indices.csv)
        junction: conditioning length (start of GEN portion)
    
    Returns:
        absolute indices in full COND+GEN array
    """
    return junction + aggressive_indices_gen

## Dual LOB Visualization (BUY & SELL)

In [ ]:
# Price conversion factor (LOBSTER prices are in 1/10000 of dollar)
PRICE_DIVISOR = 10000
N_LEVELS = 12  # Number of levels to display on each side (fixed)

def interactive_dual_lob_plot(buy_books, buy_msgs, buy_cond_lens,
                               sell_books, sell_msgs, sell_cond_lens):
    """
    Interactive dual LOB visualization: BUY on top, SELL on bottom.
    Synchronized sample selection and time slider.
    Uses absolute prices as X-axis coordinates for stable bar positions.
    Shows t-1 and t books side by side. Always displays N_LEVELS on each side.
    """
    common_sids = sorted(set(buy_books.keys()) & set(sell_books.keys()))
    
    if not common_sids:
        print("No common samples between BUY and SELL!")
        return
    
    # Controls
    id_dd = widgets.Dropdown(options=common_sids, description="Sample ID:")
    time_slider = widgets.IntSlider(min=1, max=1, step=1, description="t:")
    btn_prev = widgets.Button(description="←", layout=widgets.Layout(width="50px"))
    btn_next = widgets.Button(description="→", layout=widgets.Layout(width="50px"))
    btn_junction = widgets.Button(description="→ Junction", layout=widgets.Layout(width="100px"))
    btn_next_aggr = widgets.Button(description="→ Next Aggr", layout=widgets.Layout(width="100px"))
    
    # Info boxes
    info_box_buy = widgets.HTML()
    info_box_sell = widgets.HTML()
    
    # Create dual figure: 2 rows x 3 cols (t-1 book, t book, volumes)
    fig = make_subplots(
        rows=2, cols=3,
        subplot_titles=["BUY: Book t-1", "BUY: Book t", "BUY: Volumes",
                        "SELL: Book t-1", "SELL: Book t", "SELL: Volumes"],
        column_widths=[0.35, 0.35, 0.3],
        vertical_spacing=0.12,
        horizontal_spacing=0.05
    )
    
    # Bar styling
    bar_width = 0.008
    text_font = dict(size=11)
    
    # BUY row (row 1)
    # Col 1: t-1 book (always gray)
    fig.add_trace(go.Bar(x=[], y=[], name="BUY t-1", marker_color='gray',
                         width=bar_width, text=[], textposition='outside',
                         textfont=text_font), row=1, col=1)
    # Midprice line for t-1
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Mid t-1',
                             line=dict(color='black', width=2, dash='dash')), row=1, col=1)
    
    # Col 2: t book
    fig.add_trace(go.Bar(x=[], y=[], name="BUY t", text=[], textposition='outside',
                         width=bar_width, textfont=text_font), row=1, col=2)
    # Midprice line for t
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Mid t',
                             line=dict(color='black', width=2, dash='dash')), row=1, col=2)
    
    # Col 3: volumes
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Bid', line=dict(color='green')), row=1, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Ask', line=dict(color='red')), row=1, col=3)
    
    # SELL row (row 2)
    # Col 1: t-1 book (always gray)
    fig.add_trace(go.Bar(x=[], y=[], name="SELL t-1", marker_color='gray',
                         width=bar_width, text=[], textposition='outside',
                         textfont=text_font), row=2, col=1)
    # Midprice line for t-1
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Mid t-1',
                             line=dict(color='black', width=2, dash='dash')), row=2, col=1)
    
    # Col 2: t book
    fig.add_trace(go.Bar(x=[], y=[], name="SELL t", text=[], textposition='outside',
                         width=bar_width, textfont=text_font), row=2, col=2)
    # Midprice line for t
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Mid t',
                             line=dict(color='black', width=2, dash='dash')), row=2, col=2)
    
    # Col 3: volumes
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Bid', line=dict(color='green')), row=2, col=3)
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Ask', line=dict(color='red')), row=2, col=3)
    
    # Add cursor and junction lines for volume plots (col 3)
    for row in [1, 2]:
        xref = f"x{3 + (row-1)*3}" if row > 1 else "x3"
        fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref=xref, yref="paper",
                      line=dict(color="gray", width=1, dash="dash"))
        fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref=xref, yref="paper",
                      line=dict(color="red", width=3))
    
    fig.update_layout(width=1400, height=800, showlegend=False, template='plotly_white',
                      margin=dict(l=30, r=30, t=50, b=30))
    
    # Update axes labels
    for col in [1, 2]:
        fig.update_xaxes(title_text="Price ($)", row=1, col=col)
        fig.update_xaxes(title_text="Price ($)", row=2, col=col)
        fig.update_yaxes(title_text="Qty", row=1, col=col)
        fig.update_yaxes(title_text="Qty", row=2, col=col)
    
    fig_widget = go.FigureWidget(fig)
    
    # Cache
    _cache = {'buy_vol': {}, 'sell_vol': {}, 'buy_aggr': {}, 'sell_aggr': {}}
    
    def get_aggressive_indices_absolute(cond_lens, sid, aggressive_indices_gen):
        """Get absolute indices of aggressive orders (junction + gen_index)."""
        junction = cond_lens[sid]
        return get_absolute_aggressive_indices(aggressive_indices_gen, junction)
    
    def format_message(msg_row, is_aggressive):
        m = msg_row.astype(int)
        et_map = {1: "Limit", 2: "PartialCancel", 3: "Delete", 4: "Execution"}
        dr_map = {1: "Buy", -1: "Sell"}
        price_usd = m[4] / PRICE_DIVISOR
        info = f"{et_map.get(m[1], '?')} | {dr_map.get(m[5], '?')} | size={m[3]} | ${price_usd:.2f}"
        if is_aggressive:
            return f"<span style='color:red;font-weight:bold'>AGGRESSIVE</span> {info}"
        return info
    
    def extract_prices_and_quantities(book_row, n_levels=N_LEVELS):
        """Extract prices and quantities from orderbook row.
        LOBSTER format: [ask_price1, ask_qty1, bid_price1, bid_qty1, ask_price2, ...]
        Returns: (bid_prices, bid_qtys, ask_prices, ask_qtys) - each array of n_levels values
        Pads with zeros if fewer levels available.
        """
        ask_prices_raw = book_row[0::4]
        ask_qtys_raw = book_row[1::4]
        bid_prices_raw = book_row[2::4]
        bid_qtys_raw = book_row[3::4]
        
        # Pad to n_levels
        def pad_array(arr, n, fill_value=0):
            if len(arr) >= n:
                return arr[:n]
            return np.concatenate([arr, np.full(n - len(arr), fill_value)])
        
        # For prices, we need to extrapolate if missing
        # Use tick size of 0.01 USD = 100 in LOBSTER units
        tick = 100
        
        def pad_prices(prices, n, direction):
            """Pad prices array. direction: 1 for asks (increasing), -1 for bids (decreasing)"""
            if len(prices) >= n:
                return prices[:n]
            result = np.zeros(n)
            result[:len(prices)] = prices
            last_price = prices[-1] if len(prices) > 0 else 0
            for i in range(len(prices), n):
                result[i] = last_price + direction * tick * (i - len(prices) + 1)
            return result
        
        ask_prices = pad_prices(ask_prices_raw, n_levels, direction=1)
        bid_prices = pad_prices(bid_prices_raw, n_levels, direction=-1)
        ask_qtys = pad_array(ask_qtys_raw, n_levels, fill_value=0)
        bid_qtys = pad_array(bid_qtys_raw, n_levels, fill_value=0)
        
        return bid_prices, bid_qtys, ask_prices, ask_qtys
    
    def update_sample(*_):
        sid = id_dd.value
        max_t = min(buy_books[sid].shape[0], sell_books[sid].shape[0]) - 1
        time_slider.min = 1
        time_slider.max = max_t
        time_slider.value = 1
        
        # Cache volumes
        if sid not in _cache['buy_vol']:
            _cache['buy_vol'][sid] = compute_queued_volumes(buy_books[sid])
        if sid not in _cache['sell_vol']:
            _cache['sell_vol'][sid] = compute_queued_volumes(sell_books[sid])
        # Cache aggressive indices (absolute)
        if sid not in _cache['buy_aggr']:
            _cache['buy_aggr'][sid] = get_aggressive_indices_absolute(buy_cond_lens, sid, AGGRESSIVE_INDICES_BUY)
        if sid not in _cache['sell_aggr']:
            _cache['sell_aggr'][sid] = get_aggressive_indices_absolute(sell_cond_lens, sid, AGGRESSIVE_INDICES_SELL)
        
        # Update volume traces
        buy_bid, buy_ask, _ = _cache['buy_vol'][sid]
        sell_bid, sell_ask, _ = _cache['sell_vol'][sid]
        xs = np.arange(len(buy_bid))
        
        buy_junction = buy_cond_lens[sid]
        sell_junction = sell_cond_lens[sid]
        
        with fig_widget.batch_update():
            # BUY volumes (traces 4, 5)
            fig_widget.data[4].x = xs
            fig_widget.data[4].y = buy_bid
            fig_widget.data[5].x = xs
            fig_widget.data[5].y = buy_ask
            # SELL volumes (traces 10, 11)
            fig_widget.data[10].x = xs[:len(sell_bid)]
            fig_widget.data[10].y = sell_bid
            fig_widget.data[11].x = xs[:len(sell_ask)]
            fig_widget.data[11].y = sell_ask
            # Junction lines
            fig_widget.layout.shapes[1].x0 = buy_junction
            fig_widget.layout.shapes[1].x1 = buy_junction
            fig_widget.layout.shapes[3].x0 = sell_junction
            fig_widget.layout.shapes[3].x1 = sell_junction
        
        update_plot()
    
    def update_plot(*_):
        sid = id_dd.value
        t = time_slider.value
        
        buy_arr = buy_books[sid]
        sell_arr = sell_books[sid]
        
        # Extract prices and quantities for t-1 and t (always N_LEVELS on each side)
        buy_bid_p0, buy_bid_q0, buy_ask_p0, buy_ask_q0 = extract_prices_and_quantities(buy_arr[t-1])
        buy_bid_p1, buy_bid_q1, buy_ask_p1, buy_ask_q1 = extract_prices_and_quantities(buy_arr[t])
        sell_bid_p0, sell_bid_q0, sell_ask_p0, sell_ask_q0 = extract_prices_and_quantities(sell_arr[t-1])
        sell_bid_p1, sell_bid_q1, sell_ask_p1, sell_ask_q1 = extract_prices_and_quantities(sell_arr[t])
        
        # Convert prices to USD for X-axis
        buy_bid_x0 = buy_bid_p0 / PRICE_DIVISOR
        buy_ask_x0 = buy_ask_p0 / PRICE_DIVISOR
        buy_bid_x1 = buy_bid_p1 / PRICE_DIVISOR
        buy_ask_x1 = buy_ask_p1 / PRICE_DIVISOR
        
        sell_bid_x0 = sell_bid_p0 / PRICE_DIVISOR
        sell_ask_x0 = sell_ask_p0 / PRICE_DIVISOR
        sell_bid_x1 = sell_bid_p1 / PRICE_DIVISOR
        sell_ask_x1 = sell_ask_p1 / PRICE_DIVISOR
        
        # Compute midprices in USD
        buy_mid0 = (buy_ask_p0[0] + buy_bid_p0[0]) / 2 / PRICE_DIVISOR
        buy_mid1 = (buy_ask_p1[0] + buy_bid_p1[0]) / 2 / PRICE_DIVISOR
        sell_mid0 = (sell_ask_p0[0] + sell_bid_p0[0]) / 2 / PRICE_DIVISOR
        sell_mid1 = (sell_ask_p1[0] + sell_bid_p1[0]) / 2 / PRICE_DIVISOR
        
        buy_bid1_usd = buy_bid_p1[0] / PRICE_DIVISOR
        buy_ask1_usd = buy_ask_p1[0] / PRICE_DIVISOR
        sell_bid1_usd = sell_bid_p1[0] / PRICE_DIVISOR
        sell_ask1_usd = sell_ask_p1[0] / PRICE_DIVISOR
        
        # Create X coordinates (prices) and Y coordinates (quantities)
        # Order: Bid_n, ..., Bid1, Ask1, ..., Ask_n (sorted by price: low to high)
        buy_x0 = np.concatenate([buy_bid_x0[::-1], buy_ask_x0])
        buy_y0 = np.concatenate([-np.abs(buy_bid_q0[::-1]), np.abs(buy_ask_q0)])
        buy_x1 = np.concatenate([buy_bid_x1[::-1], buy_ask_x1])
        buy_y1 = np.concatenate([-np.abs(buy_bid_q1[::-1]), np.abs(buy_ask_q1)])
        
        sell_x0 = np.concatenate([sell_bid_x0[::-1], sell_ask_x0])
        sell_y0 = np.concatenate([-np.abs(sell_bid_q0[::-1]), np.abs(sell_ask_q0)])
        sell_x1 = np.concatenate([sell_bid_x1[::-1], sell_ask_x1])
        sell_y1 = np.concatenate([-np.abs(sell_bid_q1[::-1]), np.abs(sell_ask_q1)])
        
        # Create labels showing level number (always N_LEVELS on each side)
        labels = [f"B-{N_LEVELS-i}" for i in range(N_LEVELS)] + [f"A+{i+1}" for i in range(N_LEVELS)]
        
        # Colors based on quantity diff (for t plot)
        def get_colors(q0, q1):
            diff = np.abs(q1) - np.abs(q0)
            return ['gray' if abs(d) < 1e-8 else ('red' if d > 0 else 'blue') for d in diff]
        
        # Determine common x-axis range for both t-1 and t
        all_buy_x = np.concatenate([buy_x0, buy_x1])
        all_sell_x = np.concatenate([sell_x0, sell_x1])
        buy_x_min, buy_x_max = all_buy_x.min(), all_buy_x.max()
        sell_x_min, sell_x_max = all_sell_x.min(), all_sell_x.max()
        buy_x_margin = (buy_x_max - buy_x_min) * 0.05
        sell_x_margin = (sell_x_max - sell_x_min) * 0.05
        
        # Determine common y-axis range
        buy_y_max = max(np.abs(buy_y0).max(), np.abs(buy_y1).max(), 1) * 1.3
        sell_y_max = max(np.abs(sell_y0).max(), np.abs(sell_y1).max(), 1) * 1.3
        
        with fig_widget.batch_update():
            # BUY t-1 book (traces 0, 1) - always gray
            fig_widget.data[0].x = buy_x0
            fig_widget.data[0].y = buy_y0
            fig_widget.data[0].text = labels
            fig_widget.data[0].marker.color = 'gray'
            fig_widget.data[1].x = [buy_mid0, buy_mid0]
            fig_widget.data[1].y = [-buy_y_max, buy_y_max]
            
            # BUY t book (traces 2, 3)
            fig_widget.data[2].x = buy_x1
            fig_widget.data[2].y = buy_y1
            fig_widget.data[2].text = labels
            fig_widget.data[2].marker.color = get_colors(buy_y0, buy_y1)
            fig_widget.data[3].x = [buy_mid1, buy_mid1]
            fig_widget.data[3].y = [-buy_y_max, buy_y_max]
            
            # SELL t-1 book (traces 6, 7) - always gray
            fig_widget.data[6].x = sell_x0
            fig_widget.data[6].y = sell_y0
            fig_widget.data[6].text = labels
            fig_widget.data[6].marker.color = 'gray'
            fig_widget.data[7].x = [sell_mid0, sell_mid0]
            fig_widget.data[7].y = [-sell_y_max, sell_y_max]
            
            # SELL t book (traces 8, 9)
            fig_widget.data[8].x = sell_x1
            fig_widget.data[8].y = sell_y1
            fig_widget.data[8].text = labels
            fig_widget.data[8].marker.color = get_colors(sell_y0, sell_y1)
            fig_widget.data[9].x = [sell_mid1, sell_mid1]
            fig_widget.data[9].y = [-sell_y_max, sell_y_max]
            
            # Cursor lines for volume plots
            fig_widget.layout.shapes[0].x0 = t
            fig_widget.layout.shapes[0].x1 = t
            fig_widget.layout.shapes[2].x0 = t
            fig_widget.layout.shapes[2].x1 = t
            
            # Update x-axis range (same for t-1 and t)
            fig_widget.layout.xaxis.range = [buy_x_min - buy_x_margin, buy_x_max + buy_x_margin]
            fig_widget.layout.xaxis2.range = [buy_x_min - buy_x_margin, buy_x_max + buy_x_margin]
            fig_widget.layout.xaxis4.range = [sell_x_min - sell_x_margin, sell_x_max + sell_x_margin]
            fig_widget.layout.xaxis5.range = [sell_x_min - sell_x_margin, sell_x_max + sell_x_margin]
            
            # Update y-axis range
            fig_widget.layout.yaxis.range = [-buy_y_max, buy_y_max]
            fig_widget.layout.yaxis2.range = [-buy_y_max, buy_y_max]
            fig_widget.layout.yaxis4.range = [-sell_y_max, sell_y_max]
            fig_widget.layout.yaxis5.range = [-sell_y_max, sell_y_max]
        
        # Info boxes
        buy_junction = buy_cond_lens[sid]
        sell_junction = sell_cond_lens[sid]
        
        buy_seg = "COND" if t < buy_junction else "GEN"
        sell_seg = "COND" if t < sell_junction else "GEN"
        
        # Check if current time step (t) corresponds to an aggressive order
        # Message at t corresponds to book state at t (book after message)
        # Aggressive indices are book indices where aggressive order was applied
        buy_is_aggr = t in _cache['buy_aggr'].get(sid, [])
        sell_is_aggr = t in _cache['sell_aggr'].get(sid, [])
        
        msg_idx = t - 1
        buy_msg_text = format_message(buy_msgs[sid][msg_idx], buy_is_aggr) if msg_idx < len(buy_msgs[sid]) else "N/A"
        sell_msg_text = format_message(sell_msgs[sid][msg_idx], sell_is_aggr) if msg_idx < len(sell_msgs[sid]) else "N/A"
        
        info_box_buy.value = (
            f"<b style='color:blue'>BUY</b> | {buy_seg} | junction@{buy_junction} | "
            f"<b>Mid=${buy_mid1:.2f}</b> | Bid1=${buy_bid1_usd:.2f} Ask1=${buy_ask1_usd:.2f} | {buy_msg_text}"
        )
        info_box_sell.value = (
            f"<b style='color:red'>SELL</b> | {sell_seg} | junction@{sell_junction} | "
            f"<b>Mid=${sell_mid1:.2f}</b> | Bid1=${sell_bid1_usd:.2f} Ask1=${sell_ask1_usd:.2f} | {sell_msg_text}"
        )
    
    def on_prev(_):
        if time_slider.value > time_slider.min:
            time_slider.value -= 1
    
    def on_next(_):
        if time_slider.value < time_slider.max:
            time_slider.value += 1
    
    def on_junction(_):
        sid = id_dd.value
        junction = buy_cond_lens[sid]
        if time_slider.min <= junction <= time_slider.max:
            time_slider.value = junction
    
    def on_next_aggressive(_):
        sid = id_dd.value
        buy_aggr = _cache['buy_aggr'].get(sid, np.array([]))
        sell_aggr = _cache['sell_aggr'].get(sid, np.array([]))
        all_aggr = sorted(set(buy_aggr) | set(sell_aggr))
        current_t = time_slider.value
        
        for aggr_t in all_aggr:
            if aggr_t > current_t:
                time_slider.value = aggr_t
                return
        if all_aggr:
            time_slider.value = all_aggr[0]
    
    id_dd.observe(lambda _: update_sample(), names='value')
    time_slider.observe(lambda _: update_plot(), names='value')
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_junction.on_click(on_junction)
    btn_next_aggr.on_click(on_next_aggressive)
    
    update_sample()
    
    controls = widgets.HBox([id_dd, btn_prev, btn_next, btn_junction, btn_next_aggr, time_slider])
    display(widgets.HTML("<h3>Dual LOB: BUY (top) vs SELL (bottom) - Absolute Price X-axis</h3>"))
    display(controls)
    display(fig_widget)
    display(info_box_buy)
    display(info_box_sell)


interactive_dual_lob_plot(buy_books, buy_msgs, buy_cond_lens,
                          sell_books, sell_msgs, sell_cond_lens)

## Midprice Comparison: BUY vs SELL*(-1) vs Combined

In [5]:
def interactive_midprice_comparison(buy_books, sell_books, buy_cond_lens, sell_cond_lens):
    """
    Interactive midprice comparison for individual samples:
    - BUY midprice return
    - SELL*(-1) midprice return
    - Combined: (BUY + SELL*(-1)) / 2
    """
    common_sids = sorted(set(buy_books.keys()) & set(sell_books.keys()))
    
    id_dd = widgets.Dropdown(options=common_sids, description="Sample ID:")
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='BUY',
                             line=dict(color='blue', width=2)))
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='SELL*(-1)',
                             line=dict(color='red', width=2)))
    fig.add_trace(go.Scatter(x=[], y=[], mode='lines', name='Combined',
                             line=dict(color='purple', width=3)))
    
    # Pre-create shapes (will update positions dynamically)
    # Shape 0: horizontal line at y=0
    fig.add_shape(type="line", x0=0, x1=1, y0=0, y1=0, xref="paper", yref="y",
                  line=dict(color="gray", width=1, dash="dash"))
    # Shape 1: vertical junction line
    fig.add_shape(type="line", x0=0, x1=0, y0=0, y1=1, xref="x", yref="paper",
                  line=dict(color="gray", width=2))
    
    fig.update_layout(
        title='Midprice Return: BUY vs SELL*(-1) vs Combined',
        xaxis_title='Time Step',
        yaxis_title='Midprice Return',
        width=1000, height=500,
        template='plotly_white',
        legend=dict(orientation="h", yanchor="bottom", y=1.02)
    )
    
    fig_widget = go.FigureWidget(fig)
    info_box = widgets.HTML()
    
    def update_plot(*_):
        sid = id_dd.value
        
        buy_mid = compute_midprice(buy_books[sid])
        sell_mid = compute_midprice(sell_books[sid])
        
        # Returns from junction
        buy_junction = buy_cond_lens[sid]
        sell_junction = sell_cond_lens[sid]
        junction = max(buy_junction, sell_junction)
        
        min_len = min(len(buy_mid), len(sell_mid))
        
        # Compute returns from junction point
        buy_return = buy_mid[:min_len] - buy_mid[junction]
        sell_return = sell_mid[:min_len] - sell_mid[junction]
        sell_neg = -sell_return  # SELL * (-1)
        combined = (buy_return + sell_neg) / 2
        
        steps = np.arange(min_len)
        
        with fig_widget.batch_update():
            fig_widget.data[0].x = steps
            fig_widget.data[0].y = buy_return
            fig_widget.data[1].x = steps
            fig_widget.data[1].y = sell_neg
            fig_widget.data[2].x = steps
            fig_widget.data[2].y = combined
            fig_widget.layout.title.text = f'Midprice Return - Sample {sid}'
            # Update junction line position
            fig_widget.layout.shapes[1].x0 = junction
            fig_widget.layout.shapes[1].x1 = junction
        
        info_box.value = (
            f"<b>Sample {sid}</b> | Junction: {junction}<br>"
            f"Final BUY return: <span style='color:blue'><b>{buy_return[-1]:.0f}</b></span> | "
            f"Final SELL*(-1): <span style='color:red'><b>{sell_neg[-1]:.0f}</b></span> | "
            f"Final Combined: <span style='color:purple'><b>{combined[-1]:.0f}</b></span>"
        )
    
    id_dd.observe(update_plot, names='value')
    update_plot()
    
    display(widgets.VBox([id_dd, fig_widget, info_box]))


interactive_midprice_comparison(buy_books, sell_books, buy_cond_lens, sell_cond_lens)

## Aggregated Midprice Return: BUY vs SELL*(-1) vs Combined

In [ ]:
def plot_aggregated_midprice_combined(buy_books, sell_books, buy_cond_lens, sell_cond_lens,
                                       aggressive_indices_buy=None, aggressive_indices_sell=None):
    """
    Aggregated midprice return across all samples:
    - BUY mean ± std
    - SELL*(-1) mean ± std
    - Combined: (BUY + SELL*(-1)) / 2
    """
    common_sids = sorted(set(buy_books.keys()) & set(sell_books.keys()))
    
    # Find min length and common junction
    min_len = min(
        min(buy_books[sid].shape[0] for sid in common_sids),
        min(sell_books[sid].shape[0] for sid in common_sids)
    )
    junction = max(buy_cond_lens[common_sids[0]], sell_cond_lens[common_sids[0]])
    
    buy_returns = []
    sell_neg_returns = []
    
    for sid in common_sids:
        buy_mid = compute_midprice(buy_books[sid][:min_len])
        sell_mid = compute_midprice(sell_books[sid][:min_len])
        
        buy_ret = buy_mid - buy_mid[junction]
        sell_ret = sell_mid - sell_mid[junction]
        
        buy_returns.append(buy_ret)
        sell_neg_returns.append(-sell_ret)  # SELL * (-1)
    
    buy_arr = np.stack(buy_returns)
    sell_neg_arr = np.stack(sell_neg_returns)
    combined_arr = (buy_arr + sell_neg_arr) / 2
    
    buy_mean, buy_std = buy_arr.mean(axis=0), buy_arr.std(axis=0)
    sell_neg_mean, sell_neg_std = sell_neg_arr.mean(axis=0), sell_neg_arr.std(axis=0)
    combined_mean, combined_std = combined_arr.mean(axis=0), combined_arr.std(axis=0)
    
    steps = np.arange(min_len)
    
    fig = go.Figure()
    
    # BUY: shaded area
    fig.add_trace(go.Scatter(
        x=np.concatenate([steps, steps[::-1]]),
        y=np.concatenate([buy_mean + buy_std, (buy_mean - buy_std)[::-1]]),
        fill='toself', fillcolor='rgba(0, 0, 255, 0.1)',
        line=dict(color='rgba(255,255,255,0)'), showlegend=False
    ))
    
    # SELL*(-1): shaded area
    fig.add_trace(go.Scatter(
        x=np.concatenate([steps, steps[::-1]]),
        y=np.concatenate([sell_neg_mean + sell_neg_std, (sell_neg_mean - sell_neg_std)[::-1]]),
        fill='toself', fillcolor='rgba(255, 0, 0, 0.1)',
        line=dict(color='rgba(255,255,255,0)'), showlegend=False
    ))
    
    # Combined: shaded area
    fig.add_trace(go.Scatter(
        x=np.concatenate([steps, steps[::-1]]),
        y=np.concatenate([combined_mean + combined_std, (combined_mean - combined_std)[::-1]]),
        fill='toself', fillcolor='rgba(128, 0, 128, 0.15)',
        line=dict(color='rgba(255,255,255,0)'), showlegend=False
    ))
    
    # Mean lines
    fig.add_trace(go.Scatter(x=steps, y=buy_mean, mode='lines', name='BUY Mean',
                             line=dict(color='blue', width=2)))
    fig.add_trace(go.Scatter(x=steps, y=sell_neg_mean, mode='lines', name='SELL*(-1) Mean',
                             line=dict(color='red', width=2)))
    fig.add_trace(go.Scatter(x=steps, y=combined_mean, mode='lines', name='Combined Mean',
                             line=dict(color='purple', width=3)))
    
    # Reference lines
    fig.add_hline(y=0, line_dash="dash", line_color="gray")
    fig.add_vline(x=junction, line_color="gray", line_width=2,
                  annotation_text="Junction", annotation_position="top")
    
    # Add aggressive order markers using indices from file
    if aggressive_indices_buy is not None and len(aggressive_indices_buy) > 0:
        # Convert GEN-relative indices to absolute indices
        aggr_absolute = get_absolute_aggressive_indices(aggressive_indices_buy, junction)
        
        for i, aggr_t in enumerate(aggr_absolute):
            if aggr_t < min_len:
                fig.add_vline(x=aggr_t, line_color="green", line_width=1.5, opacity=0.5,
                              annotation_text="" if i > 0 else "Aggr", annotation_position="top")
    
    fig.update_layout(
        title='Aggregated Midprice Return: BUY vs SELL*(-1) vs Combined (Mean ± 1 Std)',
        xaxis_title='Time Step',
        yaxis_title='Midprice Return',
        width=1100, height=600,
        template='plotly_white',
        legend=dict(x=1, y=1, xanchor='right')
    )
    
    fig.show()
    
    print(f"Samples: {len(common_sids)}, Steps: {min_len}, Junction: {junction}")
    print(f"Final BUY return: {buy_mean[-1]:.2f} ± {buy_std[-1]:.2f}")
    print(f"Final SELL*(-1) return: {sell_neg_mean[-1]:.2f} ± {sell_neg_std[-1]:.2f}")
    print(f"Final Combined return: {combined_mean[-1]:.2f} ± {combined_std[-1]:.2f}")
    
    if aggressive_indices_buy is not None and len(aggressive_indices_buy) > 0:
        aggr_absolute = get_absolute_aggressive_indices(aggressive_indices_buy, junction)
        print(f"Aggressive orders at steps: {list(aggr_absolute)}")


plot_aggregated_midprice_combined(buy_books, sell_books, buy_cond_lens, sell_cond_lens,
                                   AGGRESSIVE_INDICES_BUY, AGGRESSIVE_INDICES_SELL)

## Most Volatile Samples

Анализ экстремальных значений для BUY и SELL. Скопируй sample_id в `EXCLUDE_SAMPLES_BUY` или `EXCLUDE_SAMPLES_SELL`.

In [7]:
def compute_volatility_stats(books, cond_lens, scenario_name):
    """Compute volatility statistics for one scenario."""
    stats = []
    
    for sid in books.keys():
        junction = cond_lens[sid]
        mid = compute_midprice(books[sid])
        cont = mid[junction:]
        
        stats.append({
            'sample_id': sid,
            'scenario': scenario_name,
            'range': cont.max() - cont.min(),
            'std': np.std(np.diff(cont)) if len(cont) > 1 else 0,
            'return': mid[-1] - mid[junction],
            'start_price': mid[junction],
            'end_price': mid[-1],
        })
    
    return pd.DataFrame(stats)


def compute_combined_stats(buy_books, sell_books, buy_cond_lens, sell_cond_lens):
    """Compute combined statistics."""
    common_sids = sorted(set(buy_books.keys()) & set(sell_books.keys()))
    stats = []
    
    for sid in common_sids:
        buy_junction = buy_cond_lens[sid]
        sell_junction = sell_cond_lens[sid]
        junction = max(buy_junction, sell_junction)
        
        buy_mid = compute_midprice(buy_books[sid])
        sell_mid = compute_midprice(sell_books[sid])
        
        min_len = min(len(buy_mid), len(sell_mid))
        
        buy_ret = buy_mid[:min_len] - buy_mid[junction]
        sell_neg = -(sell_mid[:min_len] - sell_mid[junction])
        combined = (buy_ret + sell_neg) / 2
        
        stats.append({
            'sample_id': sid,
            'buy_return': buy_ret[-1],
            'sell_neg_return': sell_neg[-1],
            'combined_return': combined[-1],
            'abs_combined': abs(combined[-1]),  # For sorting
            'buy_range': buy_mid[junction:min_len].max() - buy_mid[junction:min_len].min(),
            'sell_range': sell_mid[junction:min_len].max() - sell_mid[junction:min_len].min(),
        })
    
    return pd.DataFrame(stats)


# Compute stats
buy_vol_df = compute_volatility_stats(buy_books, buy_cond_lens, "BUY")
sell_vol_df = compute_volatility_stats(sell_books, sell_cond_lens, "SELL")
combined_df = compute_combined_stats(buy_books, sell_books, buy_cond_lens, sell_cond_lens)

print("=" * 80)
print("TOP 20 MOST VOLATILE BUY SAMPLES (by range)")
print("=" * 80)
print(buy_vol_df.nlargest(20, 'range')[['sample_id', 'range', 'return', 'std']].to_string(index=False))

print("\n" + "=" * 80)
print("TOP 20 MOST VOLATILE SELL SAMPLES (by range)")
print("=" * 80)
print(sell_vol_df.nlargest(20, 'range')[['sample_id', 'range', 'return', 'std']].to_string(index=False))

print("\n" + "=" * 80)
print("TOP 20 LARGEST COMBINED RETURNS (by |combined_return|)")
print("=" * 80)
top_combined = combined_df.nlargest(20, 'abs_combined')[['sample_id', 'combined_return', 'buy_return', 'sell_neg_return']]
print(top_combined.to_string(index=False))

TOP 20 MOST VOLATILE BUY SAMPLES (by range)
 sample_id  range  return       std
       279  750.0  -700.0 15.263251
        79  550.0   400.0 14.886228
       383  450.0   350.0 13.677419
      1804  400.0     0.0 11.801515
      8249  350.0  -350.0 11.461304
     10913  350.0   300.0 11.771892
      4154  300.0  -200.0  9.124423
      5302  300.0  -100.0 14.925268
      7959  300.0   150.0 11.495103
      8092  300.0  -300.0 10.522466
      9222  300.0  -250.0 10.858152
     10220  300.0  -250.0 14.193834
     11861  300.0   300.0  6.409712
     17642  300.0   300.0 11.771892
     18784  300.0  -300.0 11.771892
     19028  300.0  -250.0 10.196660
       248  250.0  -150.0 13.187878
       302  250.0   -50.0 12.092153
      3075  250.0  -250.0  8.724490
      4665  250.0  -150.0 13.187878

TOP 20 MOST VOLATILE SELL SAMPLES (by range)
 sample_id  range  return       std
       248  550.0  -450.0 11.434193
     12320  550.0   550.0 12.562643
     11861  450.0   450.0  9.431757
     17642

In [8]:
# Find outliers
OUTLIER_THRESHOLD = 1000

print("=" * 80)
print(f"BUY OUTLIERS (|return| > {OUTLIER_THRESHOLD})")
print("=" * 80)
buy_outliers = buy_vol_df[buy_vol_df['return'].abs() > OUTLIER_THRESHOLD].sort_values('return', key=abs, ascending=False)
print(f"Found {len(buy_outliers)} outliers:")
if len(buy_outliers) > 0:
    print(buy_outliers[['sample_id', 'return', 'range', 'std']].to_string(index=False))
    print(f"\nEXCLUDE_SAMPLES_BUY = {buy_outliers['sample_id'].tolist()}")

print("\n" + "=" * 80)
print(f"SELL OUTLIERS (|return| > {OUTLIER_THRESHOLD})")
print("=" * 80)
sell_outliers = sell_vol_df[sell_vol_df['return'].abs() > OUTLIER_THRESHOLD].sort_values('return', key=abs, ascending=False)
print(f"Found {len(sell_outliers)} outliers:")
if len(sell_outliers) > 0:
    print(sell_outliers[['sample_id', 'return', 'range', 'std']].to_string(index=False))
    print(f"\nEXCLUDE_SAMPLES_SELL = {sell_outliers['sample_id'].tolist()}")

print("\n" + "=" * 80)
print(f"COMBINED OUTLIERS (|combined_return| > {OUTLIER_THRESHOLD})")
print("=" * 80)
combined_outliers = combined_df[combined_df['combined_return'].abs() > OUTLIER_THRESHOLD].sort_values('combined_return', key=abs, ascending=False)
print(f"Found {len(combined_outliers)} outliers:")
if len(combined_outliers) > 0:
    print(combined_outliers[['sample_id', 'combined_return', 'buy_return', 'sell_neg_return']].to_string(index=False))
    print(f"\n# Exclude from both:")
    print(f"EXCLUDE_SAMPLES_BUY = {combined_outliers['sample_id'].tolist()}")
    print(f"EXCLUDE_SAMPLES_SELL = {combined_outliers['sample_id'].tolist()}")

BUY OUTLIERS (|return| > 1000)
Found 0 outliers:

SELL OUTLIERS (|return| > 1000)
Found 0 outliers:

COMBINED OUTLIERS (|combined_return| > 1000)
Found 0 outliers:


## Market Impact Beta Calculation (Combined)

In [ ]:
def compute_market_impact_beta_combined(buy_books, buy_msgs, buy_cond_lens,
                                         sell_books, sell_msgs, sell_cond_lens,
                                         aggressive_indices_buy, aggressive_indices_sell,
                                         iteration_filter=None):
    """
    Compute market impact beta using combined data from BUY and SELL scenarios.
    For SELL, we flip the sign of impact.
    
    Args:
        aggressive_indices_buy: array of GEN-relative indices for BUY aggressive orders
        aggressive_indices_sell: array of GEN-relative indices for SELL aggressive orders
        iteration_filter: None = all iterations, or int = only that iteration (1-indexed)
    """
    eps = 1e-12
    common_sids = sorted(set(buy_books.keys()) & set(sell_books.keys()))
    
    points = []  # (x, y, sample_id, iteration, scenario)
    alphas = []
    
    for scenario, books, msgs, cond_lens, aggr_indices_gen in [
        ('BUY', buy_books, buy_msgs, buy_cond_lens, aggressive_indices_buy),
        ('SELL', sell_books, sell_msgs, sell_cond_lens, aggressive_indices_sell)
    ]:
        if len(aggr_indices_gen) == 0:
            print(f"Warning: No aggressive indices for {scenario}")
            continue
            
        for sid in common_sids:
            if sid not in books:
                continue
                
            msg_arr = msgs[sid]
            book_arr = books[sid]
            junction = cond_lens[sid]

            # Convert GEN-relative indices to absolute indices
            aggr_indices = get_absolute_aggressive_indices(aggr_indices_gen, junction)
            
            # Filter to valid indices
            aggr_indices = aggr_indices[aggr_indices < len(msg_arr)]
            
            if len(aggr_indices) < 2:
                continue

            # Get message data at aggressive order positions
            sizes = msg_arr[aggr_indices, 3].astype(float)
            prices = msg_arr[aggr_indices, 4].astype(float)

            first_idx = aggr_indices[0]
            ref_price = (book_arr[first_idx, 0] + book_arr[first_idx, 2]) / 2

            if ref_price <= 0:
                continue

            Q_cum = np.cumsum(sizes)
            notional_cum = np.cumsum(sizes * prices)
            vwap = notional_cum / np.maximum(Q_cum, eps)
            
            # For BUY: vwap > ref_price (positive impact)
            # For SELL: vwap < ref_price, but we want positive impact too
            if scenario == 'BUY':
                impact = (vwap - ref_price) / ref_price
            else:  # SELL
                impact = (ref_price - vwap) / ref_price  # Flip sign for SELL
            
            impact = np.abs(impact)  # Take absolute value

            gen_msgs = msg_arr[junction:]
            exec_mask = gen_msgs[:, 1].astype(int) == 4
            V_exp = np.sum(gen_msgs[exec_mask, 3].astype(float)) if np.any(exec_mask) else 1.0
            V_exp = max(V_exp, eps)

            midprices = (book_arr[junction:, 0] + book_arr[junction:, 2]) / 2
            H, L = np.max(midprices), np.min(midprices)
            
            if H > L and L > 0:
                eta = np.log(H / L) / 0.8325546
                alphas.append(np.log(max(eta, eps)))

            valid_mask = impact > eps
            if np.sum(valid_mask) < 2:
                continue

            x_vals = np.log(Q_cum[valid_mask] / V_exp)
            y_vals = np.log(impact[valid_mask])
            iterations = np.arange(1, len(sizes) + 1)[valid_mask]
            
            for x, y, it in zip(x_vals, y_vals, iterations):
                points.append((x, y, sid, int(it), scenario))

    if not points or not alphas:
        print("No valid data for beta estimation")
        return

    # Convert and filter
    all_X = np.array([p[0] for p in points])
    all_Y = np.array([p[1] for p in points])
    all_sids = np.array([p[2] for p in points])
    all_iters = np.array([p[3] for p in points])
    all_scenarios = np.array([p[4] for p in points])
    
    if iteration_filter is not None:
        mask = all_iters == iteration_filter
        X = all_X[mask]
        Y = all_Y[mask]
        sids = all_sids[mask]
        iters = all_iters[mask]
        scenarios = all_scenarios[mask]
    else:
        X, Y, sids, iters, scenarios = all_X, all_Y, all_sids, all_iters, all_scenarios
    
    alpha_global = np.mean(alphas)
    y_adj = Y - alpha_global
    valid = np.isfinite(X) & np.isfinite(y_adj) & (X != 0)
    
    if np.sum(valid) < 2:
        print("Not enough valid points")
        return
        
    beta_ols = float(np.dot(X[valid], y_adj[valid]) / np.dot(X[valid], X[valid]))

    # Plot
    fig = go.Figure()
    
    # Separate BUY and SELL points
    buy_mask = scenarios == 'BUY'
    sell_mask = scenarios == 'SELL'
    
    fig.add_trace(go.Scatter(
        x=X[buy_mask], y=Y[buy_mask], mode='markers',
        marker=dict(size=7, opacity=0.6, color='blue'),
        name='BUY'
    ))
    
    fig.add_trace(go.Scatter(
        x=X[sell_mask], y=Y[sell_mask], mode='markers',
        marker=dict(size=7, opacity=0.6, color='red'),
        name='SELL'
    ))

    x_range = np.linspace(X.min(), X.max(), 100)
    fig.add_trace(go.Scatter(
        x=x_range, y=alpha_global + beta_ols * x_range,
        mode='lines', line=dict(color='purple', width=3),
        name=f'OLS: β = {beta_ols:.3f}'
    ))

    fig.add_trace(go.Scatter(
        x=x_range, y=alpha_global + 0.5 * x_range,
        mode='lines', line=dict(color='black', width=3, dash='dash'),
        name='Theory: β = 0.5'
    ))

    iter_text = f"iter={iteration_filter}" if iteration_filter else "all iterations"
    fig.update_layout(
        title=f'Combined Market Impact: β_OLS = {beta_ols:.4f} ({iter_text})',
        xaxis_title='log(Q / V_exp)',
        yaxis_title='log(Impact)',
        width=1100, height=600,
        template='plotly_white',
        hovermode='closest'
    )
    fig.show()

    print(f"Points: {len(X)} (BUY: {np.sum(buy_mask)}, SELL: {np.sum(sell_mask)})")
    print(f"Alpha: {alpha_global:.4f}, Beta: {beta_ols:.4f}")
    return beta_ols


print("=== Combined Market Impact (all iterations) ===")
beta_combined = compute_market_impact_beta_combined(
    buy_books, buy_msgs, buy_cond_lens,
    sell_books, sell_msgs, sell_cond_lens,
    AGGRESSIVE_INDICES_BUY, AGGRESSIVE_INDICES_SELL
)

print("\n=== Combined Market Impact (last iteration only) ===")
beta_combined_last = compute_market_impact_beta_combined(
    buy_books, buy_msgs, buy_cond_lens,
    sell_books, sell_msgs, sell_cond_lens,
    AGGRESSIVE_INDICES_BUY, AGGRESSIVE_INDICES_SELL,
    iteration_filter=5
)